# Nonlinear Spatial Models

This guide demonstrates three nonlinear spatial models in `bayespecon` on synthetic data.

## Spatial Probit

For binary outcomes $y_i \in \{0,1\}$, latent utilities $y^*$ follow a spatial lag:

$$y^* = \rho W y^* + X\beta + a + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

where $a$ are region-level random effects and $y_i = \mathbf{1}[y^*_i > 0]$.

## SAR Tobit

For censored outcomes, the spatial autoregressive Tobit specifies:

$$y^* = \rho W y^* + X\beta + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I)$$
$$y_i = \max(c,\, y^*_i)$$

where $c$ is the censoring threshold (default 0).

## SEM Tobit

The spatial error Tobit places spatial dependence in the disturbance:

$$y^* = X\beta + u, \quad u = \lambda W u + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I)$$
$$y_i = \max(c,\, y^*_i)$$

In [1]:
import numpy as np
from libpysal.graph import Graph
from bayespecon import SpatialProbit, SARTobit, SEMTobit, dgp

rng = np.random.default_rng(1234)

In [2]:
# SpatialProbit
m, n_per_region = 8, 20

# Create line-graph weights: each node connected to neighbors
focal, neighbor = [], []
for i in range(m):
    if i > 0:
        focal.append(i)
        neighbor.append(i - 1)
    if i < m - 1:
        focal.append(i)
        neighbor.append(i + 1)
Wm = Graph.from_arrays(np.array(focal), np.array(neighbor), weight=np.ones(len(focal))).transform('r')

rho, beta, sigma_a = 0.35, np.array([0.3, 1.0]), 0.8

sp_data = dgp.simulate_spatial_probit(
    W=Wm,
    rho=rho,
    beta=beta,
    sigma_a=sigma_a,
    n_per_region=n_per_region,
    rng=rng,
 )

sp = SpatialProbit(
    y=sp_data["y"],
    X=sp_data["X"],
    W=sp_data["W_graph"],
    region_ids=sp_data["region_ids"],
)
sp.fit(draws=300, tune=300, chains=2, random_seed=42, progressbar=False)
sp.summary(var_names=["rho", "beta", "sigma_a"])

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [rho, beta, sigma_a, a_raw]
Sampling 2 chains for 300 tune and 300 draw iterations (600 + 600 draws total) took 1 seconds.
There were 4 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
rho,0.186,0.336,-0.414,0.774,0.025,0.014,184.0,281.0,1.01
x0,0.386,0.964,-1.416,2.500,0.068,0.052,199.0,293.0,1.01
x1,1.040,0.211,0.646,1.403,0.011,0.009,396.0,400.0,1.00
sigma_a,1.823,0.599,0.966,3.062,0.044,0.032,179.0,300.0,1.01


In [ ]:
# SARTobit and SEMTobit
n = 30

# Create line-graph weights: each node connected to neighbors
focal, neighbor = [], []
for i in range(n):
    if i > 0:
        focal.append(i)
        neighbor.append(i - 1)
    if i < n - 1:
        focal.append(i)
        neighbor.append(i + 1)
W = Graph.from_arrays(np.array(focal), np.array(neighbor), weight=np.ones(len(focal))).transform('r')

beta = np.array([1.0, 1.5])
sigma, rho, lam = 0.8, 0.4, 0.35

sar_data = dgp.simulate_sar_tobit(
    W=W,
    rho=rho,
    beta=beta,
    sigma=sigma,
    censoring=0.0,
    rng=rng,
)
sar_tobit = SARTobit(
    y=sar_data["y"],
    X=sar_data["X"],
    W=sar_data["W_graph"],
    censoring=0.0,
)
sar_tobit.fit(draws=300, tune=300, chains=2, random_seed=42, progressbar=False)

sem_data = dgp.simulate_sem_tobit(
    W=W,
    lam=lam,
    beta=beta,
    sigma=sigma,
    censoring=0.0,
    rng=rng,
)
sem_tobit = SEMTobit(
    y=sem_data["y"],
    X=sem_data["X"],
    W=sem_data["W_graph"],
    censoring=0.0,
)
sem_tobit.fit(draws=300, tune=300, chains=2, random_seed=42, progressbar=False)

sar_tobit.summary(var_names=["rho", "beta", "sigma"]), sem_tobit.summary(var_names=["lam", "beta", "sigma"])

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [rho, beta, sigma, y_cens_gap]
Sampling 2 chains for 300 tune and 300 draw iterations (600 + 600 draws total) took 0 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [lam, beta, sigma, y_cens_gap]
Sampling 2 chains for 300 tune and 300 draw iterations (600 + 600 draws total) took 0 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 f

(        mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  \
 rho    0.454  0.084   0.299    0.605      0.004    0.003     510.0     415.0   
 x0     0.778  0.214   0.394    1.185      0.010    0.008     495.0     300.0   
 x1     1.653  0.173   1.341    1.999      0.009    0.008     387.0     272.0   
 sigma  0.817  0.142   0.596    1.104      0.006    0.006     498.0     421.0   
 
        r_hat  
 rho      1.0  
 x0       1.0  
 x1       1.0  
 sigma    1.0  ,
         mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  \
 lam    0.428  0.153   0.138    0.703      0.007    0.006     444.0     431.0   
 x0     1.440  0.208   1.060    1.861      0.009    0.009     511.0     389.0   
 x1     1.223  0.106   1.030    1.441      0.006    0.005     320.0     315.0   
 sigma  0.557  0.083   0.415    0.706      0.004    0.003     415.0     430.0   
 
        r_hat  
 lam     1.01  
 x0      1.00  
 x1      1.00  
 sigma   1.00  )

: 